# Initialisation

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import colors, cm
from astropy.table import Table, join
from astropy.cosmology import Planck18
from astropy import units as u
from time import time

## Read data

In [ ]:
#besta_dr1 = Table.read("data/BESTA/20260316/merged_catalogue.fits", unit_parse_strict='silent')
besta_dr1 = Table.read("data/BESTA/stacked_catalogue.fits", unit_parse_strict='silent')

## Redshift bins

In [ ]:
zz = np.linspace(redshift_bins[0], redshift_bins[-1], 101)
cosmic_time = Planck18.age(zz)
main_sequence = -np.log10(cosmic_time.to_value(u.yr))

In [ ]:
redshift_bins = np.linspace(0.05, 1.95, 20)
redshift_bins = np.interp(np.arange(4, 13, 1)<<u.Gyr, cosmic_time[::-1], zz[::-1])[::-1]
redshift_centre = (redshift_bins[:-1] + redshift_bins[1:]) / 2
#redshift_colour = ['cyan', 'blue', 'green', 'orange', 'red']
redshift_labels = [f'z={z0:.2f}' for z0 in redshift_centre]

In [ ]:
plt.plot(zz, cosmic_time.to_value(u.Gyr), 'k-')
for edge in redshift_bins:
    plt.axvline(edge, c='k', ls='-', alpha=.2)
    plt.axhline(Planck18.age(edge).to_value(u.Gyr), c='k', ls='-', alpha=.2)
plt.ylabel("cosmic time [Gyr]")
plt.xlabel("redshift")


In [ ]:
redshift_bins

# BESTA vs PHZ

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(6, 4), squeeze=False)

ax = axes[0, 0]
ax.set_ylabel("BESTA mean $z$")
ax.set_xlabel("PHZ median $z$")
hh = ax.hist2d(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['Z__mean'], bins=redshift_bins, norm=colors.LogNorm(vmin=1), cmap="nipy_spectral")
ax.plot([0., 1.9], [.1, 2.], 'k:')
ax.plot([.1, 2.], [.1, 2.], 'k--')
ax.plot([.1, 2.], [0., 1.9], 'k:')
plt.colorbar(hh[-1], ax=ax, label="number of galaxies per bin")

In [ ]:
good_z = np.where((besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'] - besta_dr1['Z__mean'])**2 < .2**2)[0]
bad_z = np.where((besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'] - besta_dr1['Z__mean'])**2 > .2**2)[0]
print(f'{good_z.size}/{len(besta_dr1)} good redshifts = {100*good_z.size/len(besta_dr1):.2f}%')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey='row', gridspec_kw={'wspace': 0, 'hspace': 0}, squeeze=False)

ax = axes[0, 0]
ax.hist(besta_dr1['Z__mean'], bins=redshift_bins, color='k', histtype='step')
ax.hist([besta_dr1['Z__mean'][good_z], besta_dr1['Z__mean'][bad_z]], bins=redshift_bins, color=['b', 'r'], alpha=.25, histtype='barstacked')

ax = axes[0, 1]
ax.hist(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], bins=redshift_bins, color='k', histtype='step')
ax.hist([besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'][good_z], besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'][bad_z]], bins=redshift_bins, color=['b', 'r'], alpha=.25, histtype='barstacked')

ax = axes[1, 0]
ax.set_ylabel("BESTA stellar mass [M$_\odot$]")
ax.set_ylim(7.5, 12.5)
ax.set_xlabel("BESTA mean $z$")
ax.scatter(besta_dr1['Z__mean'], besta_dr1['stellar_mass__mean'], s=1, alpha=.005, color='k')
#ax.scatter(besta_dr1['Z__mean'][good_z], besta_dr1['stellar_mass__mean'][good_z], s=1, alpha=.01, color='k')
#ax.scatter(besta_dr1['Z__mean'][bad_z], besta_dr1['stellar_mass__mean'][bad_z], s=1, alpha=.05, color='k')

ax = axes[1, 1]
ax.set_xlabel("PHZ median $z$")
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['stellar_mass__mean'], s=1, alpha=.005, color='k')
#ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'][good_z], besta_dr1['stellar_mass__mean'][good_z], s=1, alpha=.01, color='k')
#ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'][bad_z], besta_dr1['stellar_mass__mean'][bad_z], s=1, alpha=.05, color='k')

# M(z) and sSFR(z)

## Mass bins

In [ ]:
log_mass_bins = np.arange(6.5, 14.6, 0.5)

In [ ]:
log_mass_threshold_z1 = 10.
log_mass_massive = 10.5

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), width_ratios=(1, .05),
                         gridspec_kw={'wspace': 0, 'hspace': 0}, squeeze=False)

norm = colors.LogNorm(vmin=1)
cmap = 'nipy_spectral'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

ax = axes[0, 0]
ax.set_ylabel('log ( M / M$_\odot$ )')
hh = ax.hist2d(besta_dr1['Z__mean'], besta_dr1['stellar_mass__mean'],
               bins=(redshift_bins, log_mass_bins), norm=norm, cmap=cmap)

plt.colorbar(hh[-1], cax=axes[0, -1], label="number of galaxies per bin")

'''

norm = colors.Normalize(vmin=8.75, vmax=11.25)
cmap = 'nipy_spectral'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

ax = axes[1, 0]
ax.set_ylabel("log ( sSFR8 / yr$^{-1}$ )")
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['log_ssfr_8p0__mean'], c=besta_dr1['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

#plt.colorbar(cm, cax=axes[1, -1], label="log ( sSFR9 / yr$^{-1}$ )")
plt.colorbar(cm, cax=axes[1, -1], label="log ( M / M$_\odot$ )")


ax = axes[2, 0]
ax.set_ylabel("log ( sSFR9 / yr$^{-1}$ )")
ax.set_ylim(-13.5, -8.5)
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['log_ssfr_9p0__mean'], c=besta_dr1['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

plt.colorbar(cm, cax=axes[2, -1], label="log ( M / M$_\odot$ )")
'''


for ax in axes[:, :-1].ravel():
    for i, edge in enumerate(redshift_bins):
        ax.axvline(edge, c='k', ls=':')
    ax.set_xlim(-.01, redshift_bins[-1]+.01)

for ax in axes[0, :-1].ravel():
    ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'k-', lw=3)
    ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'w--')
    ax.axhline(log_mass_massive, c='w', lw=3)
    ax.axhline(log_mass_massive, c='k', ls=':')
    for i, edge in enumerate(redshift_bins):
        if i>0:
            bin_threshold = log_mass_threshold_z1 + 2*np.log10(edge)
            ax.plot([edge, redshift_bins[i-1]], [bin_threshold, bin_threshold], 'k:')

for ax in axes[1:, :-1].ravel():
    ax.plot(zz, main_sequence, 'k-', lw=3)
    ax.plot(zz, main_sequence, 'w--')

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 8), width_ratios=(1, 1, 1, .05),
                         gridspec_kw={'wspace': 0, 'hspace': 0}, squeeze=False)

norm = colors.Normalize(vmin=0, vmax=3.75)
cmap = 'rainbow'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

ax = axes[0, 0]
ax.set_title('specz')
ax.set_ylabel('log ( M / M$_\odot$ )')
ax.set_ylim(8.5, 12.5)
#ax.scatter(specz['Z'], specz['stellar_mass__mean'], c=specz['bestfit_chi2'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[0, 1]
ax.set_title('photoz')
ax.yaxis.set_ticklabels('')
ax.set_ylim(8.5, 12.5)
#ax.scatter(photoz['PHZ_PP_MEDIAN_REDSHIFT'], photoz['stellar_mass__mean'], c=photoz['bestfit_chi2'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[0, 2]
ax.set_title('DR1')
ax.yaxis.set_ticklabels('')
ax.set_ylim(8.5, 12.5)
#ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['stellar_mass__mean'], c=besta_dr1['bestfit_chi2'], norm=norm, cmap=cmap, s=1, alpha=.25)
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

plt.colorbar(cm, cax=axes[0, -1], label="best-fit $\chi^2$")


norm = colors.Normalize(vmin=8.75, vmax=11.25)
cmap = 'nipy_spectral'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

ax = axes[1, 0]
ax.set_ylabel("log ( sSFR8 / yr$^{-1}$ )")
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('spectroscopic z')
#ax.scatter(specz['Z'], specz['log_ssfr_8p0__mean'], c=specz['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[1, 1]
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
#ax.scatter(photoz['PHZ_PP_MEDIAN_REDSHIFT'], photoz['log_ssfr_8p0__mean'], c=photoz['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[1, 2]
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['log_ssfr_8p0__mean'], c=besta_dr1['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

#plt.colorbar(cm, cax=axes[1, -1], label="log ( sSFR9 / yr$^{-1}$ )")
plt.colorbar(cm, cax=axes[1, -1], label="log ( M / M$_\odot$ )")


ax = axes[2, 0]
ax.set_ylabel("log ( sSFR9 / yr$^{-1}$ )")
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('spectroscopic z')
#ax.scatter(specz['Z'], specz['log_ssfr_9p0__mean'], c=specz['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[2, 1]
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
#ax.scatter(photoz['PHZ_PP_MEDIAN_REDSHIFT'], photoz['log_ssfr_9p0__mean'], c=photoz['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[2, 2]
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['log_ssfr_9p0__mean'], c=besta_dr1['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

plt.colorbar(cm, cax=axes[2, -1], label="log ( M / M$_\odot$ )")


for ax in axes[:, :-1].ravel():
    for i, edge in enumerate(redshift_bins):
        ax.axvline(edge, c='k', ls=':')
    ax.set_xlim(-.01, 1.01*redshift_bins[-1])

for ax in axes[0, :-1].ravel():
    ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'k-', lw=3)
    ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'w--')
    for i, edge in enumerate(redshift_bins):
        if i>0:
            bin_threshold = log_mass_threshold_z1 + 2*np.log10(edge)
            ax.plot([edge, redshift_bins[i-1]], [bin_threshold, bin_threshold], 'k:')

for ax in axes[1:, :-1].ravel():
    ax.plot(zz, main_sequence, 'k-', lw=3)
    ax.plot(zz, main_sequence, 'w--')

## Mass function

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'hspace': 0, 'wspace': 0}, squeeze=False, sharex=True)

def plot_distributions(plot_column, title, table, redshit_column):
    ax = axes[:, plot_column]
    ax[0].set_title(title)
    for i, z0 in enumerate(redshift_centre):
        if log_mass_threshold_z1 + 2*np.log10(redshift_bins[i+1]) > log_mass_massive:
            break
        galaxies = np.where(
            (table[redshit_column] > redshift_bins[i])
            & (table[redshit_column] <= redshift_bins[i+1])
            & (table['stellar_mass__mean'] > log_mass_threshold_z1 + 2*np.log10(redshift_bins[i+1]))
        )
        log_mass = np.sort(table['stellar_mass__mean'][galaxies])
        mass_above = np.cumsum(10**log_mass[::-1])
        n11 = log_mass.size - np.searchsorted(log_mass, log_mass_massive)
        ax[0].plot(log_mass[::-1], np.arange(log_mass.size)/n11, label=f'z={z0:.2f}', c=plt.cm.nipy_spectral(z0/redshift_bins[-1]))
        ax[1].plot(log_mass[::-1], mass_above/mass_above[n11], label=f'z={z0:.2f}', c=plt.cm.nipy_spectral(z0/redshift_bins[-1]))
        ax[0].legend()
        if plot_column == 0:
            ax[0].set_ylabel(f"N(M) / N($10^{{{log_mass_massive}}}$ M$_\odot$)")
            ax[0].set_ylim(3e-4, 3e2)
            ax[1].set_ylim(3e-2, 3e1)
            ax[0].set_yscale('log')
            ax[1].set_ylabel(f"M(M) / M($10^{{{log_mass_massive}}}$ M$_\odot$)")
            ax[1].set_yscale('log')
        ax[1].set_xlabel("log( M / M$_\odot$ )")
        ax[1].set_xlim(7.75, 15.25)
        ax[0].grid(alpha=.2)
        ax[1].grid(alpha=.2)

#plot_distributions(0, 'specz', specz, 'Z')
#plot_distributions(1, 'photoz', photoz, 'PHZ_PP_MEDIAN_REDSHIFT')
plot_distributions(0, 'DR1', besta_dr1, 'Z__mean')

# Main sequence

In [ ]:
n_bins = redshift_centre.size
norm = colors.Normalize(vmin=-1., vmax=1.)
#cmap = 'nipy_spectral'
cmap = 'RdYlBu'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

def plot_main_sequence(joint_table, redshift_column):
    fig, axes = plt.subplots(n_bins+1, 2, sharey=True, sharex=True, figsize=(12, n_bins*3), gridspec_kw={'wspace': 0, 'hspace': 0}, squeeze=False)
    for i, z0 in enumerate(redshift_centre):
        galaxies = np.where((joint_table[redshift_column] > redshift_bins[i]) & (joint_table[redshift_column] <= redshift_bins[i+1]))
        mass = joint_table['stellar_mass__mean'][galaxies]
        ssfr8 = joint_table['log_ssfr_8p0__mean'][galaxies]
        ssfr9 = joint_table['log_ssfr_9p0__mean'][galaxies]
        ax = axes[i, 0]
        sc = ax.scatter(mass, ssfr8, s=1, alpha=.5, c=ssfr8-ssfr9, norm=norm, cmap=cmap)
        ax.axvline(log_mass_threshold_z1+2*np.log10(redshift_bins[i+1]), c='k', ls=':')
        ax.axhline(np.interp(z0, zz, main_sequence), c='k', lw=3, label=f'z={z0:.2f}')
        ax.axhline(np.interp(z0, zz, main_sequence), c='w', ls='--')
        ax.grid(c='k', alpha=.2)
        ax = axes[i, 1]
        sc = ax.scatter(mass, ssfr9, s=1, alpha=.5, c=ssfr8-ssfr9, norm=norm, cmap=cmap)
        ax.axvline(log_mass_threshold_z1+2*np.log10(redshift_bins[i+1]), c='k', ls=':')
        ax.axhline(np.interp(z0, zz, main_sequence), c='k', lw=3, label=f'z={z0:.2f}')
        ax.axhline(np.interp(z0, zz, main_sequence), c='w', ls='--')
        ax.legend(loc='lower right')
        ax.grid(c='k', alpha=.2)
    ax.set_xlim(7.8, 12.2)
    ax.set_ylim(-13.5, -8.5)
    #axes[0, 0].set_ylabel("log( sSFR8 / yr )")
    #axes[1, 0].set_ylabel("log( sSFR9 / yr )")
    axes[0, 0].set_title(r"log( sSFR8 [yr$^{-1}$] )")
    axes[0, 1].set_title(r"log( sSFR9 [yr$^{-1}$] )")
    for ax in axes[-2, :]:
        ax.set_xlabel("log( M / M$_\odot$ )")
    for ax in axes[-1, :]:
        ax.set_axis_off()
    cb = plt.colorbar(cm, ax=axes[-1, :], label='log( sSFR8 / sSFR9 )', orientation="horizontal")
    plt.savefig("MS.png")
    #plt.close()

In [ ]:
#plot_main_sequence(specz, 'Z')
#plot_main_sequence(photoz, "PHZ_PP_MEDIAN_REDSHIFT")

#plot_main_sequence(besta_dr1, "PHZ_PP_MEDIAN_REDSHIFT")
t0 = time()
plot_main_sequence(besta_dr1, "Z__mean")
print(f'{time()-t0:.2f}')